# Cervello-Bambino — training su Colab

Notebook per eseguire le fasi di training intensive (PyTorch, `cervello/`) su GPU Colab invece che sul PC locale (solo CPU).

Prima di lanciare le celle: **Runtime -> Cambia tipo di runtime -> GPU**.

## 1. Verifica GPU

In [ ]:
!nvidia-smi

## 2. Clona il repository

In [ ]:
REPO_URL = "https://github.com/Mecks3D/llm-test.git"

!rm -rf llm-test
!git clone $REPO_URL
%cd llm-test

## 3. Installa le dipendenze

In [ ]:
!pip install -q -r requirements.txt

## 4. Verifica che torch veda la GPU

In [ ]:
import torch
print("CUDA disponibile:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 5. Monta Google Drive (per salvare i checkpoint)

La sessione Colab perde tutto alla scadenza: i checkpoint vanno salvati su Drive, non nel filesystem locale del runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/cervello-bambino/checkpoint"
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoint dir:", CHECKPOINT_DIR)

## 6. Smoke test: la suite di test passa nell'ambiente Colab?

Prima di lanciare training lunghi, verifica che l'ambiente sia equivalente a quello locale.

In [ ]:
!python -m pytest -q

## 6b. Canary di apprendimento (Fase 2a, gruppo 6)

Test "cancello duro": 32 esempi di stadio 1, overfit fino a esattezza-risposta 100% (max 1000 step). Marcato `slow` (escluso dalla suite di default): su CPU e' impraticabile, va eseguito qui con GPU.

In [ ]:
!python -m pytest -q -m slow -s


## 6c. Run di fumo (Fase 2a, tappa T6)

Prima del training vero: genera il dataset ridotto (200 storie, solo stadio 1) e allena per 300 step con `configs/v1_fumo.yaml`. Verifica che la loss scenda e che compaiano log/checkpoint in `dati/risultati/v1_fumo/`. Le prime valutazioni possono essere lente: il modello appena inizializzato non ha ancora imparato a emettere `[FINE]`, quindi la decodifica greedy può arrivare fino al tetto `ctx=3072` prima di fermarsi. Se una singola valutazione impiega più di qualche minuto, interrompi e segnalalo.

In [ ]:
!python -m esami.genera --config configs/v1_fumo.yaml --stadio 1


In [ ]:
!python -m cervello.addestra --config configs/v1_fumo.yaml --stadio 1


In [ ]:
import json, shutil, os

with open('dati/risultati/v1_fumo/log.jsonl', encoding='utf-8') as f:
    righe = [json.loads(r) for r in f]
for r in righe:
    print(r)

# Esito dell'esame (esattezza per tipo + conteggi di calibrazione:
# 'errore' = struttura giusta ma contenuto sbagliato -> poco training;
# 'malformata' dominante -> problema di decodifica, indagare).
with open('dati/risultati/v1_fumo/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Il filesystem di Colab e' effimero: copia i risultati su Drive.
dest = os.path.join(CHECKPOINT_DIR, 'v1_fumo')
shutil.copytree('dati/risultati/v1_fumo', dest, dirs_exist_ok=True)
print('risultati copiati su Drive ->', dest)


## 7. Training (curriculum completo, stadi 1-3)

Esegue `esami/genera.py` + `cervello/addestra.py` con `configs/v1.yaml` (i default di produzione). Il seed è nel config (`seed_torch`). Prima di lanciarlo: verifica l'esito e i token/sec del run di fumo (sezione 6c) e la stima di durata del curriculum completo — se supera ~2 ore, chiedi conferma prima di procedere (FASE2_PIANO.md, decisione 10). Genera prima i dataset degli stadi 1-3 con `esami/genera.py` se non esistono ancora.

In [ ]:
!python -m cervello.addestra --config configs/v1.yaml
